# **<center><span style= "color:#2F539B;">Data Classes</span></center>**

## ***<span style= "color:purple;">Imports </span>***

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
!pip install kagglehub[pandas-datasets]

In [ ]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = ""

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "zalando-research/fashionmnist",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

## ***<span style= "color:orange;">Dataset </span>***

In [3]:
import pandas as pd
df = pd.read_csv("datasets/fashion-mnist_train.csv")

In [4]:
torch.manual_seed(42)

In [5]:
df.head(5)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [8]:
X_train = X_train/255.0
X_test_test = X_test/255.0

### ***<span style= "color:#85BB65;"> creating custom dataset class </span>***

In [9]:
class CustomDataset(Dataset):
    def __init__(self, features, labels):

        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [10]:
train_dataset = CustomDataset(X_train, y_train)
len(train_dataset)

48000

In [11]:
test_dataset = CustomDataset(X_test, y_test)

In [12]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32, shuffle=False)

### ***<span style= "color:#85BB65;"> Neural Network class </span>***

In [14]:
class MyNN(nn.Module):
    
    def __init__(self, num_features):

        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features,128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,10)
        )
    
    def forward(self,x):
        return self.model(x)

In [15]:
epochs = 10
learning_rate = 0.1


In [16]:
model = MyNN(X_train.shape[1])

#loss function
criterion = nn.CrossEntropyLoss()

#optimizer
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

## ***<span style= "color:orange;">Model </span>***

In [17]:
for epoch in range(epochs):

    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:

        #forwardpass
        outputs = model(batch_features)

        #calcuate loss
        loss = criterion(outputs, batch_labels)

        # backpass
        optimizer.zero_grad()
        loss.backward()

        #update grads
        optimizer.step()

        total_epoch_loss = total_epoch_loss + loss.item()
    
    avg_loss = total_epoch_loss/len(train_loader)
    print(f'Epoch: {epoch + 1}, Loss: {avg_loss}')

Epoch: 1, Loss: 0.6351939875682195
Epoch: 2, Loss: 0.42953018795947234
Epoch: 3, Loss: 0.38687779793143273
Epoch: 4, Loss: 0.3592242692609628
Epoch: 5, Loss: 0.3368441379517317
Epoch: 6, Loss: 0.32270959017798306
Epoch: 7, Loss: 0.30773366042474903
Epoch: 8, Loss: 0.29594149482250215
Epoch: 9, Loss: 0.2863469832663735
Epoch: 10, Loss: 0.2742569016168515


### ***<span style= "color:#85BB65;">Model evaluation </span>***


In [18]:
model.eval()

MyNN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [20]:
# useful variables
total = 0
correct = 0

with torch.no_grad():
    
    for batch_features, batch_labels in test_loader:

        outputs = model(batch_features)

        _, predicted = torch.max(outputs,1)

        total = total + batch_labels.shape[0]

        correct = correct + (predicted == batch_labels).sum().item()



        

In [22]:
#acurracy
print(correct/total)

0.8341666666666666
